# 06 — Evaluación: análisis de fallos, comparación pareada y olvido catastrófico

Notebook de la **Fase 4** (herramientas de evaluación, PLAN.md §29–§38 y §47–§48).

**Honestidad de escala:** todo lo ejecutado aquí es **CPU / SMOKE**. Los datos reales que se analizan son los del smoke `e0_smoke` (modelo 0.5B, 5 registros de validación) y el smoke de olvido (0.5B + adapter de 10 pasos). Las comparaciones reales E0 vs E1 con el modelo 7B siguen **pendientes de Kaggle**; este notebook demuestra que las herramientas para leerlas ya existen y funcionan.


## (a) Análisis de fallos del smoke real `e0_smoke`

Cargamos `experiments/e0_smoke/scores.json` y clasificamos cada ítem fallido con la taxonomía E1–E10 de §48 (mapeo señal del harness → código en `docs/ERROR_TAXONOMY.md`, implementado en `scripts/failure_analysis.py`).


In [1]:
import json, sys
from pathlib import Path
sys.path.insert(0, "../src")
sys.path.insert(0, "../scripts")
import failure_analysis as fa

scores = json.loads(Path("../experiments/e0_smoke/scores.json").read_text())
items = scores["items"]
counts = fa.taxonomy_counts(items)
for code_id in sorted(counts):
    print(f"{code_id}: {counts[code_id]}  — {fa.TAXONOMY[code_id]}")
print("\nÍtems fallidos:", sum(1 for i in items if fa.classify_item(i)), "de", len(items))


E1: 0  — Invalid structure — parsed JSON is not an object / top-level structure unusable
E10: 0  — Format failure / other (L1 parse failure, or any unmapped harness signal)
E2: 5  — Missing required field
E3: 0  — Wrong field type
E4: 5  — Invalid name (not kebab-case)
E5: 0  — Incomplete file set (files empty, missing SKILL.md, file entry incomplete)
E6: 0  — Unsafe path
E7: 0  — Empty or invalid content field (e.g. empty description)
E8: 0  — Materialization failure (L5 fails while L2 is valid)
E9: 0  — Field-fidelity failure (L2 valid but L3 name exact-match fails)

Ítems fallidos: 5 de 5


In [2]:
# Tabla/gráfico simple de conteos (solo códigos con ítems)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
present = {k: v for k, v in counts.items() if v}
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(list(present), list(present.values()))
ax.set_ylabel("ítems")
ax.set_title("Taxonomía de fallos — e0_smoke (SMOKE, n=5)")
plt.tight_layout(); plt.show()
print(present)


{'E2': 5, 'E4': 5}


**Interpretación.** El fallo dominante es la **invención de esquema**: el modelo base produce JSON válido (L1 = 1.0) pero no el Skill Blueprint pedido — en los 5 ítems faltan campos obligatorios (**E2**: `blueprint_version`, `purpose`, `when_to_use`, `when_not_to_use`, `files`) y el nombre no es kebab-case (**E4**). Es exactamente el fallo que el SFT (E1) debe corregir: no es un problema de "no sabe generar JSON", es un problema de "no conoce *nuestro* esquema". Si tras E1 los E2/E4 no caen, el fine-tuning no habrá aprendido el formato, diga lo que diga el loss.


## (b) Mecánica de `compare.py` — demo con datos SINTÉTICOS

⚠️ **El segundo archivo de esta sección es un dato sintético de demostración, no un experimento:** tomamos las predicciones reales del smoke y truncamos cada salida a 200 caracteres (una degradación artificial) para mostrar cómo se leen los conteos pareados y McNemar. Ningún número de esta sección es un resultado experimental.


In [3]:
import json
from pathlib import Path
from skillgen.evaluation.baseline import load_split_records, score_predictions
from skillgen.evaluation.compare import paired_counts, mcnemar_exact, wilson_ci, score_diff_ci

preds = [json.loads(l) for l in Path("../experiments/e0_smoke/predictions.jsonl").read_text().splitlines()]
degraded = [dict(p, output=p["output"][:200]) for p in preds]  # SINTÉTICO: truncado artificial
records = load_split_records("val", limit=5)
items_a = items  # e0_smoke real (referencia A)
items_b = score_predictions(degraded, records)["items"]  # sintético degradado (B)

for metric in ("l1_parse", "l2_valid"):
    c = paired_counts(items_a, items_b, metric=metric)
    p = mcnemar_exact(c["only_a"], c["only_b"])
    print(metric, c, f"McNemar exact p = {p:.4f}")
lo, hi, diff = score_diff_ci(items_a, items_b, metric="l1_parse")
print(f"l1_parse: diff (B-A) = {diff:+.3f}, IC95% bootstrap pareado = [{lo:+.3f}, {hi:+.3f}]")
print("Wilson IC95% tasa L1 de A:", tuple(round(x, 3) for x in wilson_ci(5, 5)))


l1_parse {'both_pass': 0, 'only_a': 5, 'only_b': 0, 'both_fail': 0, 'n': 5} McNemar exact p = 0.0625
l2_valid {'both_pass': 0, 'only_a': 0, 'only_b': 0, 'both_fail': 5, 'n': 5} McNemar exact p = 1.0000
l1_parse: diff (B-A) = -1.000, IC95% bootstrap pareado = [-1.000, -1.000]
Wilson IC95% tasa L1 de A: (0.566, 1.0)


**Cómo se lee.** En L1, los 5 ítems pasan en A y fallan en B (`only_a=5, only_b=0`): son 5 regresiones puras, y el McNemar exacto da p = 0.0625 — con n=5 ni siquiera una regresión total alcanza p<0.05, una lección de por qué el test congelado tiene 100 ítems y no 5. En L2 no hay movimiento (`both_fail=5`, p = 1.0): truncar no "rompe" nada que no estuviera ya roto. La regla de lectura (§34): **primero los conteos pareados (arreglado/roto), después el agregado**.


## (c) Olvido catastrófico — resultados reales del smoke (§37)

Base 0.5B vs el adapter smoke de la Fase 3 (LoRA, 10 pasos, 20 registros), sobre el set local original de 20 preguntas de conocimiento general de `forgetting.py`. **Este set NO es ARC** — es un set local escrito para este repo; el set ARC real se carga con `load_arc_questions()` cuando `datasets` puede descargar ARC-Challenge.


In [4]:
import json
from pathlib import Path
res = json.loads(Path("../experiments/forgetting_smoke/results.json").read_text())
print("Fuente de preguntas:", res["question_source"])
print(f"Base:    accuracy = {res['base']['accuracy']:.4f} ({res['base']['correct']}/{res['base']['n']})")
print(f"Adapter: accuracy = {res['adapter_run']['accuracy']:.4f} ({res['adapter_run']['correct']}/{res['adapter_run']['n']})")
print(f"Caída: {res['accuracy_drop_points']:.1f} puntos")
distintos = sum(1 for a, b in zip(res["base"]["per_question"], res["adapter_run"]["per_question"]) if a["predicted"] != b["predicted"])
print("Predicciones distintas entre base y adapter:", distintos)


Fuente de preguntas: local (original 20-question set, NOT ARC)
Base:    accuracy = 0.8000 (16/20)
Adapter: accuracy = 0.8000 (16/20)
Caída: 0.0 puntos
Predicciones distintas entre base y adapter: 0


**Cómo correrá §37 a escala Kaggle:** `load_arc_questions(limit=50)` cargará 50 preguntas de ARC-Challenge (etiqueta `source="arc"` en el resultado), se evaluará el modelo base 7B y después el adapter E1 real, con el mismo scoring por loglikelihood (argmax sobre las opciones, sin generación larga). Una **caída > 5% se documenta como hallazgo** (no bloquea v1, §37). En este smoke la caída es 0.0 y las predicciones son idénticas — lo esperable de un adapter de 10 pasos; lo que queda probado es el arnés antes/después.


## (d) Cierre: cómo se leerá la comparación E1 vs E0 en Kaggle

1. **Conteos pareados primero:** con `compare.py` sobre los `scores.json` de E0 y E1 (mismo split congelado, mismos prompts), mirar `only_a` (regresiones de E1) y `only_b` (arreglos) en L1/L2/L5 antes que cualquier promedio.
2. **Significancia:** McNemar exacto sobre los pares discordantes + IC 95% (Wilson por tasa, bootstrap pareado para la diferencia). Con n=100, diferencias reales de formato deben ser detectables; con n=5, como se vio, casi nada lo es.
3. **Fallos por taxonomía:** `scripts/failure_analysis.py --baseline <e0 scores>` genera el `failures.md` §47 de E1 con REGRESSION / NEW FAILURE rellenos; la hipótesis concreta es que E2/E4 (esquema inventado) colapsen.
4. **Olvido:** §37 con las 50 de ARC; caída >5% = hallazgo documentado.
5. **Puerta §39 (Accept/Reject):** E1 solo se promueve si su SkillGen Score ≥ E0 + 2 puntos **y** sin regresiones en el set experto. Si no, se rechaza aunque "el loss bajara".
